# Event segmentation per video

Notebook này đọc output mới từ `embedding_vectors_per_video.ipynb`:

- `features/<model>/<video_id>.npy`
- `map-keyframes/<video_id>.csv`

Sau đó gom keyframes liền kề thành events theo thời gian + scene similarity, rồi lưu:

- `events/<video_id>.npy`
- `map-event/<video_id>.csv`


## 1. Import

In [ ]:
from pathlib import Path
from io import StringIO
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


## 2. Config

Nếu Kaggle dataset của bạn có tên khác, chỉ cần sửa `INPUT_ROOT`.

In [ ]:
INPUT_ROOT = Path('/kaggle/input/datasets/phngtrnhng/embeddings-new')

OUTPUT_ROOT = Path('/kaggle/working/event_embeddings_per_video')
EVENT_DIR = OUTPUT_ROOT / 'events'
MAP_EVENT_DIR = OUTPUT_ROOT / 'map-event'

EVENT_DIR.mkdir(parents=True, exist_ok=True)
MAP_EVENT_DIR.mkdir(parents=True, exist_ok=True)

# Rule gom event:
# - nếu 2 keyframes cách nhau quá xa => tách event
# - nếu embedding khác cảnh quá nhiều => tách event
MAX_TIME_GAP_SEC = 6.0
SCENE_SIMILARITY_THRESHOLD = 0.72

# Giới hạn event quá dài để event không bị nuốt cả video.
# Đặt None nếu không muốn giới hạn.
MAX_EVENT_DURATION_SEC = 45.0

print('Input root:', INPUT_ROOT)
print('Output root:', OUTPUT_ROOT)


## 3. Detect input folders

In [ ]:
FEATURES_ROOT = INPUT_ROOT / 'features'
MAP_KEYFRAMES_DIR = INPUT_ROOT / 'map-keyframes'

model_dirs = sorted([p for p in FEATURES_ROOT.iterdir() if p.is_dir()])
FEATURE_DIR = model_dirs[0]

feature_files = sorted(FEATURE_DIR.glob('*.npy'))
video_ids = [p.stem for p in feature_files]

print('Feature dir:', FEATURE_DIR)
print('Map-keyframes dir:', MAP_KEYFRAMES_DIR)
print('Number of videos:', len(video_ids))
print('First videos:', video_ids[:5])


## 4. Helper đọc CSV an toàn

Cell này xử lý nhẹ các trường hợp CSV có BOM, encoding lạ hoặc dấu quote cong.

In [ ]:
def read_csv_safe(path):
    raw = Path(path).read_bytes()
    try:
        text = raw.decode('utf-8-sig')
    except UnicodeDecodeError:
        text = raw.decode('cp1252')

    text = text.replace('“', '"').replace('”', '"')
    df = pd.read_csv(StringIO(text), sep=None, engine='python')
    df.columns = [str(c).strip().strip('\ufeff"“”') for c in df.columns]

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.strip('"“”')

    return df


## 5. Test load 1 video

In [ ]:
sample_video = video_ids[0]

sample_embeddings = np.load(FEATURE_DIR / f'{sample_video}.npy').astype(np.float32)
sample_map = read_csv_safe(MAP_KEYFRAMES_DIR / f'{sample_video}.csv')

print('Sample video:', sample_video)
print('Embeddings shape:', sample_embeddings.shape)
print('Map shape:', sample_map.shape)
display(sample_map.head())


## 6. Create event embeddings per video

Ý tưởng:

- mỗi dòng trong `map-keyframes/<video_id>.csv` ứng với 1 vector trong `.npy`
- duyệt keyframe theo `pts_time`
- keyframe tiếp theo được nối vào event hiện tại nếu:
  - khoảng cách thời gian không quá `MAX_TIME_GAP_SEC`
  - cosine similarity với event hiện tại >= `SCENE_SIMILARITY_THRESHOLD`
- embedding của event = trung bình các keyframe embeddings trong event, sau đó normalize lại

In [ ]:
summary_rows = []

for video_id in tqdm(video_ids, desc='Videos'):
    feature_path = FEATURE_DIR / f'{video_id}.npy'
    map_path = MAP_KEYFRAMES_DIR / f'{video_id}.csv'

    if not map_path.exists():
        print('Missing map file:', map_path)
        continue

    embeddings = np.load(feature_path).astype(np.float32)
    key_map = read_csv_safe(map_path)

    # Đảm bảo có các cột cần thiết từ output embedding_vectors_per_video.ipynb
    key_map['n'] = key_map['n'].astype(int)
    key_map['pts_time'] = key_map['pts_time'].astype(float)
    key_map['frame_idx'] = key_map['frame_idx'].astype(int)

    # Nếu thứ tự CSV bị lệch, sort lại rồi reorder embedding theo cột n.
    order = np.argsort(key_map['pts_time'].values)
    key_map = key_map.iloc[order].reset_index(drop=True)
    embeddings = embeddings[key_map['n'].values - 1]

    # Normalize keyframe embeddings để cosine similarity = dot product.
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.maximum(norms, 1e-12)

    events = []
    event_vectors = []

    current_indices = [0]
    current_vector = embeddings[0].copy()

    for i in range(1, len(key_map)):
        prev_time = float(key_map.loc[i - 1, 'pts_time'])
        cur_time = float(key_map.loc[i, 'pts_time'])
        event_start_time = float(key_map.loc[current_indices[0], 'pts_time'])

        time_gap = cur_time - prev_time
        event_duration_if_added = cur_time - event_start_time
        similarity = float(np.dot(current_vector, embeddings[i]))

        should_split = False
        if time_gap > MAX_TIME_GAP_SEC:
            should_split = True
        if similarity < SCENE_SIMILARITY_THRESHOLD:
            should_split = True
        if MAX_EVENT_DURATION_SEC is not None and event_duration_if_added > MAX_EVENT_DURATION_SEC:
            should_split = True

        if should_split:
            idx = np.array(current_indices, dtype=int)
            event_vec = embeddings[idx].mean(axis=0)
            event_vec = event_vec / max(float(np.linalg.norm(event_vec)), 1e-12)

            start_row = key_map.iloc[current_indices[0]]
            end_row = key_map.iloc[current_indices[-1]]
            event_id = len(events)

            events.append({
                'event_id': f'{video_id}_E{event_id:04d}',
                'event_embedding_index': event_id,
                'video_id': video_id,
                'start_n': int(start_row['n']),
                'end_n': int(end_row['n']),
                'start_sec': float(start_row['pts_time']),
                'end_sec': float(end_row['pts_time']),
                'start_frame': int(start_row['frame_idx']),
                'end_frame': int(end_row['frame_idx']),
                'keyframe_ns': ' '.join(map(str, key_map.iloc[idx]['n'].astype(int).tolist())),
                'n_keyframes': int(len(idx)),
            })
            event_vectors.append(event_vec.astype(np.float32))

            current_indices = [i]
            current_vector = embeddings[i].copy()
        else:
            current_indices.append(i)
            current_vector = embeddings[current_indices].mean(axis=0)
            current_vector = current_vector / max(float(np.linalg.norm(current_vector)), 1e-12)

    # Flush event cuối cùng.
    idx = np.array(current_indices, dtype=int)
    event_vec = embeddings[idx].mean(axis=0)
    event_vec = event_vec / max(float(np.linalg.norm(event_vec)), 1e-12)

    start_row = key_map.iloc[current_indices[0]]
    end_row = key_map.iloc[current_indices[-1]]
    event_id = len(events)

    events.append({
        'event_id': f'{video_id}_E{event_id:04d}',
        'event_embedding_index': event_id,
        'video_id': video_id,
        'start_n': int(start_row['n']),
        'end_n': int(end_row['n']),
        'start_sec': float(start_row['pts_time']),
        'end_sec': float(end_row['pts_time']),
        'start_frame': int(start_row['frame_idx']),
        'end_frame': int(end_row['frame_idx']),
        'keyframe_ns': ' '.join(map(str, key_map.iloc[idx]['n'].astype(int).tolist())),
        'n_keyframes': int(len(idx)),
    })
    event_vectors.append(event_vec.astype(np.float32))

    event_embeddings = np.stack(event_vectors).astype(np.float32)
    event_map = pd.DataFrame(events)

    np.save(EVENT_DIR / f'{video_id}.npy', event_embeddings)
    event_map.to_csv(MAP_EVENT_DIR / f'{video_id}.csv', index=False)

    summary_rows.append({
        'video_id': video_id,
        'num_keyframes': int(len(key_map)),
        'num_events': int(len(event_map)),
        'embedding_dim': int(event_embeddings.shape[1]),
        'event_feature_path': str(EVENT_DIR / f'{video_id}.npy'),
        'event_map_path': str(MAP_EVENT_DIR / f'{video_id}.csv'),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUTPUT_ROOT / 'event_summary.csv', index=False)

print('Done')
display(summary.head())


## 7. Check output

In [ ]:
print('Output folders:')
print(EVENT_DIR)
print(MAP_EVENT_DIR)

print('\nNumber of event npy files:', len(list(EVENT_DIR.glob('*.npy'))))
print('Number of event map csv files:', len(list(MAP_EVENT_DIR.glob('*.csv'))))

sample_video = summary.iloc[0]['video_id']
sample_event_embeddings = np.load(EVENT_DIR / f'{sample_video}.npy')
sample_event_map = pd.read_csv(MAP_EVENT_DIR / f'{sample_video}.csv')

print('\nSample video:', sample_video)
print('Sample event embeddings shape:', sample_event_embeddings.shape)
display(sample_event_map.head())


## 8. Zip output để download hoặc tạo Kaggle Dataset

In [ ]:
!cd /kaggle/working && zip -r event_embeddings_per_video.zip event_embeddings_per_video
